<a href="https://colab.research.google.com/github/vitor-dornela/FaesaBI/blob/main/C3/ETL/despesas_vitoria_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# fonte dos dados https://transparencia.vitoria.es.gov.br/DadosAbertos.Lista.aspx

In [5]:
# Download dos arquivos - compatível com Windows e Colab
import requests
import os

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

downloads = [
    (
        'https://wstransparencia.vitoria.es.gov.br/api/despesas/xlsx?exercicio=2026&tipoDespesa=P',
        'despesas_vitoria_2026.xlsx'
    ),
    (
        'https://wstransparencia.vitoria.es.gov.br/api/despesas/csv?exercicio=2026&tipoDespesa=P',
        'despesas_vitoria_2026_P.csv'
    ),
]

for url, filename in downloads:
    print(f'Baixando {filename}...')
    response = requests.get(url, headers=HEADERS, stream=True)
    response.raise_for_status()
    with open(filename, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    size_kb = os.path.getsize(filename) / 1024
    print(f'  Salvo: {filename} ({size_kb:.1f} KB)\n')

Baixando despesas_vitoria_2026.xlsx...
  Salvo: despesas_vitoria_2026.xlsx (997.5 KB)

Baixando despesas_vitoria_2026_P.csv...
  Salvo: despesas_vitoria_2026_P.csv (10509.8 KB)



In [6]:
# Inspecionar CSV - compatível com Windows e Colab
import os

csv_file = 'despesas_vitoria_2026_P.csv'

# Tamanho do arquivo (substitui: file -i *.csv)
size_bytes = os.path.getsize(csv_file)
print(f'{csv_file}: tamanho={size_bytes} bytes, encoding=LATIN1 (inferido)')

# Primeiras 3 linhas (substitui: head -n 3 *.csv)
print()
with open(csv_file, 'r', encoding='latin1') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(line, end='')

despesas_vitoria_2026_P.csv: tamanho=10762025 bytes, encoding=LATIN1 (inferido)

Ano,ValorOrcado,ValorAutorizado,ValorReservado,ValorEmpenhado,ValorLiquidado,ValorPago,CodUnidadeOrcamentaria,UnidadeOrcamentaria,CodUnidadeGestora,UnidadeGestora,CodOrgao,Orgao,CodFuncao,Funcao,CodSubFuncao,SubFuncao,CodPrograma,Programa,CodAcao,Acao,CodFonte,Fonte,CpfCnpjFornecedor,NumeroProcesso,NumeroContrato,CodNaturezaDespesa,NaturezaDespesa,NomeFornecedor,Data,IdEmpenho,Licitacao,Descricao,NumeroEmpenho
2026,,,,"978,00","978,00","978,00",,,301               ,INST PREV ASS SERV MUN VITORIA,020000,Instituto de Previdencia de Vitória - IPAMV,09,Previdência Social,122,Administração Geral,0035,Apoio Administrativo - IPAMV,2154,Manutenção dos Serviços Administrativos,1.802.0000.0000,RPPS - TAXA DE ADMINISTRAÇÃO,30.778.880/0001-92  ,936/2025,Não informado,33903902 ,CONDOMINIOS                                       ,CONDOMINIO DO EDIFICIO BEMGE,02/01/2026 00:00:00,3141618,Não informado,Pagamento de condomín

## leitura arquivo excel

In [7]:
import pandas as pd

dfex = pd.read_excel("despesas_vitoria_2026.xlsx")
dfex.columns



C:\Users\vdmas\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Index(['NumeroPagamento', 'ProcessoPagamento', 'DigitoPagamento', 'idEmpenho',
       'NumeroEmpenho', 'NumeroLiquidacao', 'Exercicio', 'Data', 'Descricao',
       'Licitacao', 'UnidadeGestoraEmpenho', 'CodUnidadeGestoraEmpenho',
       'Valor', 'CpfCnpjFornecedor', 'NomeFornecedor'],
      dtype='str')

In [8]:
dfex.groupby("UnidadeGestoraEmpenho")['Valor'].describe()

,count,mean,std,min,25%,50%,75%,max
UnidadeGestoraEmpenho,,,,,,,,
"CDTIV - Cia de Desenvolvimento, Turismo e Inovação de Vitória",508.0,31851.257520,2.087387e+05,-104608.26,533.3500,1799.800,10040.3825,2370000.00
CENTRAL DE SERVIÇOS,1289.0,114899.720403,7.344423e+05,-1610341.10,1641.8600,9706.390,42187.5100,11998397.16
CONTROLADORIA GERAL DO MUNICIPIO,200.0,6341.146000,1.120459e+04,0.67,120.7750,1654.510,6856.6700,58642.35
CÂMARA MUNICIPAL DE VITÓRIA,651.0,35187.694255,1.283059e+05,-186793.09,415.4950,2559.990,12185.6000,1516226.01
FUNDO AMBIENTAL,59.0,14714.664915,3.078461e+04,29.16,793.5850,5674.360,17012.1900,225700.00
FUNDO DE TECNOLOGIA - FACITEC,1.0,1198.000000,NaN,1198.00,1198.0000,1198.000,1198.0000,1198.00
FUNDO FINANCEIRO,241.0,541422.748008,2.271087e+06,-16653.30,861.6100,4279.160,34936.9500,17406296.19
FUNDO MUN. INFANCIA E ADOLESCENCIA,3.0,198999.336667,1.484327e+03,197293.92,198499.0050,199704.090,199852.0450,200000.00
FUNDO MUNICIPAL DE ASSISTÊNCIA SOCIAL DE VITORIA,1026.0,30716.780097,1.392579e+05,-3036.00,1200.0000,1621.000,3946.7000,1807473.32


## Leitura e Limpeza do Arquivo CSV


In [ ]:
import pandas as pd
import warnings
import re

print("Lendo o arquivo bruto com Pandas...")
w_list = []
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    df_bruto = pd.read_csv('despesas_vitoria_2026_P.csv', engine='python', on_bad_lines='warn', encoding='latin1')
    w_list = [str(warn.message) for warn in w if issubclass(warn.category, pd.errors.ParserWarning)]

linhas_afetadas = []
for msg in w_list:
    match = re.search(r'line (\d+)', msg)
    if match:
        linhas_afetadas.append(int(match.group(1)))

print(f"Total de warnings (bad_lines): {len(linhas_afetadas)}")
if linhas_afetadas:
    print(f"Linhas afetadas (amostra de 5): {linhas_afetadas[:5]}")
    
    print("\n--- AMOSTRA DA PRIMEIRA LINHA COM PROBLEMA NO ARQUIVO BRUTO ---")
    with open('despesas_vitoria_2026_P.csv', 'r', encoding='latin1') as f:
        for i, line in enumerate(f, start=1):
            if i == linhas_afetadas[0]:
                print(f"Linha {i}:")
                print(line)
                break


In [ ]:
import csv
import io
import pandas as pd

def read_cleaned_csv(filepath):
    cleaned_rows = []
    
    with open(filepath, 'r', encoding='latin1') as f:
        reader = csv.reader(f)
        header = next(reader)
        current_row = []
        
        for row in reader:
            if not current_row:
                if not row: continue
                current_row = row
            else:
                if not row:
                    current_row[-1] += '\n'
                else:
                    current_row[-1] = current_row[-1] + '\n' + row[0]
                    current_row.extend(row[1:])
            
            if len(current_row) < 34:
                continue
                
            if len(current_row) > 34:
                excess = len(current_row) - 34
                desc_pieces = current_row[32:33+excess]
                desc = ",".join(desc_pieces).replace('"', "'")
                current_row = current_row[:32] + [desc] + [current_row[-1]]
                
            cleaned_rows.append(current_row)
            current_row = []
            
    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_MINIMAL)
    writer.writerow(header)
    writer.writerows(cleaned_rows)
    output.seek(0)
    return output

print("Executando a limpeza das aspas quebradas...")
cleaned_csv_io = read_cleaned_csv('despesas_vitoria_2026_P.csv')
print("Limpeza concluída! Dados estruturados na memória.")


In [ ]:
import pandas as pd
import warnings

print("Lendo os dados limpos com Pandas...")
w_list = []
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    cleaned_csv_io.seek(0)
    dfcsv = pd.read_csv(cleaned_csv_io, engine='python', on_bad_lines='warn')
    w_list = [str(warn.message) for warn in w if issubclass(warn.category, pd.errors.ParserWarning)]

print(f"Total de warnings (bad_lines) APÓS limpeza: {len(w_list)}")

if len(w_list) > 0:
    print("Amostra dos warnings encontrados:")
    for warn in w_list[:3]:
        print(f" - {warn}")
else:
    print("✅ Excelente! Nenhuma linha quebrou o Parser. Os dados estão perfeitos estruturalmente.")
    
# Mostra uma amostra do dataframe limpo
dfcsv.head(3)
